<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_25_iterators_advanced/note_lesson_25_iterators_advanced_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 25 — Ітератори advanced

Диспетчерська «Смачно + Таксі» отримує потік подій від кур'єрів: `picked`, `delivered`, серед них биті рядки й події попередньої зміни. Сьогодні:

1. **Клас-ітератор** з `__iter__` і `__next__`, ітерабельне проти ітератора.
2. **Генератор зсередини**: стани, `return`, `close()`, `.send()`, автомат, `yield from`.
3. **`itertools`**: `dropwhile`, `takewhile`, `groupby`, `pairwise`, `accumulate`, `combinations` та інші.
4. **ETL-конвеєр** подій зі стадіями, які легко тестувати.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | **ітератор і ітерабельне**: протокол, `for` зсередини, клас-курсор, генератор у `__iter__`, нескінченний ітератор, `iter(функція, стоп)` | 1.1–1.9 | **Задача 1** |
| 2 | **генератор зсередини**: стани, `return`, `close()` і `finally` | 2.1–2.5 | |
| 3 | **`.send()`**: сопрограма, запуск і `@primed`, автомат, `yield from` | 3.1–3.9 | **Задача 2** |
| 4 | **`itertools`**: межі потоку, групи, пари, підсумки, перебори, `chain`, `tee`, `islice` | 4.1–4.14 | **Задача 3** |
| 5 | **ETL-конвеєр**: стадії, лінивість, тести стадій, pull і push | 5.1–5.8 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (класи, функції, константи) лише створює їх — **запуск у наступній клітинці**. Генератор можна перезапустити, просто виконавши клітинку запуску ще раз.
- Перед **🔮 Прогнозом** спершу скажи, що буде, потім розгортай відповідь.
- Три **задачі** розв'язуєш сам: код — у клітинці задачі, перевірка — у наступній.
- Теорія, схеми й архітектура — у книзі: [Урок 25. Ітератори advanced](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_25/).

## 🔁 Пригадай (без підглядання)

1. Що робить `for` з об'єктом перед першим елементом?
2. Чому другий `list()` від генератора порожній?
3. Що повертав `__iter__` кошика в уроці 24?

<details>
<summary>Відповіді</summary>

1. Викликає `iter()`, далі `next()` до `StopIteration`.
2. Генератор одноразовий.
3. Готовий ітератор словника.

</details>

Журнал вечірньої зміни — з ним працюємо весь урок. Кожен рядок — «час кур'єр замовлення подія». Тут є все, з чим стикається справжній потік: події попередньої зміни (до 18:00), битий рядок, доставки різних кур'єрів упереміш.

In [ ]:
LOG = [
    "17:52 D-2 98 delivered",
    "17:58 D-1 97 delivered",
    "18:03 D-1 101 picked",
    "18:05 D-2 102 picked",
    "18:07 D-3 103 picked",
    "18:21 D-1 101 delivered",
    "18:24 ?? зламаний рядок",
    "18:29 D-2 102 delivered",
    "18:31 D-1 104 picked",
    "18:44 D-3 103 delivered",
    "18:52 D-1 104 delivered",
]

In [ ]:
print(len(LOG))

---
## 1. Ітератор і ітерабельне

### Приклад 1.1. Протокол ззовні: `iter()` і `next()`

Пригадаймо урок 11: `for` спершу просить у колекції **курсор** (`iter`), а потім бере з нього елементи (`next`), доки той не скаже «кінець».

In [ ]:
dishes = ["Борщ", "Узвар"]
cursor = iter(dishes)
print(type(cursor).__name__)
print(next(cursor))
print(next(cursor))
try:
    next(cursor)
except StopIteration:
    print("StopIteration — елементи скінчились")
print(next(cursor, "кінець"))

`iter(dishes)` повернув окремий об'єкт — `list_iterator`. Кожен `next` віддає наступний елемент; коли їх нема, `next` кидає `StopIteration`. Другий аргумент `next(cursor, "кінець")` — значення замість винятку.

### Приклад 1.2. Що робить `for` насправді

Той самий цикл, записаний вручну через `while`, `iter`, `next` і `StopIteration`.

In [ ]:
def manual_for(iterable):
    cursor = iter(iterable)
    while True:
        try:
            item = next(cursor)
        except StopIteration:
            break
        print("  тіло циклу:", item)

In [ ]:
manual_for(LOG[:3])

Саме це Python робить під капотом кожного `for`. Отже, щоб наш клас працював у `for`, йому потрібні два методи:

- `__iter__` — повернути курсор;
- `__next__` (у курсора) — віддати наступний елемент або кинути `StopIteration`.

Дві ролі, які важливо не змішувати:

```text
ітерабельне (журнал)            ітератор (курсор)
знає дані                       пам'ятає позицію
iter() → НОВИЙ курсор щоразу    iter() → сам себе
можна обходити багато разів     одноразовий

   ShiftLog ──iter()──► курсор 1 (позиція 0) ──next()──► рядок … StopIteration
            ──iter()──► курсор 2 (позиція 0)
```

### Приклад 1.3. Перша спроба: журнал сам собі курсор

🔮 **Прогноз:** що надрукує другий `list(log)`?

In [ ]:
class ShiftLog:
    def __init__(self, lines):
        self.lines = lines
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.lines):
            raise StopIteration
        line = self.lines[self.position]
        self.position += 1
        return line

In [ ]:
log = ShiftLog(LOG[2:5])
print(list(log))
print(list(log))

<details>
<summary>Відповідь</summary>

`[]`: курсор `position` живе в самому журналі й після першого проходу стоїть у кінці. Рядки нікуди не ділися — їх просто нема ким прочитати вдруге.

</details>

### Приклад 1.4. Вкладені цикли по такому журналу

🔮 **Прогноз:** три рядки, вкладений цикл «кожен з кожним». Скільки пар вийде?

In [ ]:
log = ShiftLog(LOG[2:5])
pairs = [(a[:5], b[:5]) for a in log for b in log]
print(len(pairs), pairs)

<details>
<summary>Відповідь</summary>

2, а не 9. Зовнішній цикл узяв перший рядок, внутрішній — той самий курсор — дочитав решту два. Коли зовнішній попросив наступний рядок, курсор уже стояв у кінці.

</details>

### Приклад 1.5. Розділяємо ролі: журнал і курсор

`ShiftLog` тепер лише зберігає дані, а на кожен `iter()` створює **новий** `ShiftLogIterator`.

In [ ]:
class ShiftLogIterator:
    def __init__(self, lines):
        self._lines = lines
        self._position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._position >= len(self._lines):
            raise StopIteration
        line = self._lines[self._position]
        self._position += 1
        return line


class ShiftLog:
    def __init__(self, lines):
        self.lines = list(lines)

    def __iter__(self):
        return ShiftLogIterator(self.lines)

In [ ]:
log = ShiftLog(LOG[2:20])
print(len(list(log)), len(list(log)))
print(len([(a, b) for a in log for b in log]))

`LOG[2:20]` — 9 рядків (зріз за межами списку просто закінчується). Кожен `list` отримав свій курсор — 9 і 9, а вкладений цикл — усі 9 × 9 = 81 пар. Журнал можна обходити скільки завгодно разів:

In [ ]:
log = ShiftLog(LOG)
for l in log:
    print(l)

### Приклад 1.6. Ітератор повертає себе, ітерабельне — новий курсор

In [ ]:
log = ShiftLog(LOG)
cursor = iter(log)
print(type(cursor).__name__, iter(cursor) is cursor, iter(log) is log)
print(next(cursor))
print(next(cursor))
print(next(iter(log)))

`iter(cursor) is cursor` — `True`: `__iter__` ітератора повертає `self`, тому ітератор теж можна передати в `for`. `iter(log)` — щоразу новий курсор: після двох `next(cursor)` свіжий курсор знову починає з першого рядка.

### Приклад 1.7. Генератор у `__iter__` — курсор без окремого класу

Писати клас-курсор доводиться рідко: якщо `__iter__` — генератор, Python сам створює новий генератор (тобто новий курсор) на кожен виклик.

In [ ]:
class ShiftLog:
    def __init__(self, lines):
        self.lines = list(lines)

    def __iter__(self):
        for line in self.lines:
            yield line

In [ ]:
log = ShiftLog(LOG[2:5])
print(len(list(log)), len(list(log)))
print(len([(a, b) for a in log for b in log]))
print(type(iter(log)).__name__)

Два проходи по 3 рядки й 9 пар у вкладеному циклі — як з окремим класом-курсором, але вдвічі менше коду. `iter(log)` тепер повертає `generator`.

**Правило:** для простого перебору — генератор у `__iter__`. Справжній клас-ітератор — коли курсор має **власний інтерфейс**: `peek()`, позицію для відновлення читання (Задача 1).

### Приклад 1.8. Нескінченний ітератор зі станом

Видавець номерів замовлень: 105, 106, … без кінця. `StopIteration` він не кидає ніколи, а поточний номер видно в атрибуті — його можна зберегти й відновити роботу після збою.

In [ ]:
from itertools import islice


class OrderNumbers:
    def __init__(self, start):
        self.next_number = start

    def __iter__(self):
        return self

    def __next__(self):
        number = self.next_number
        self.next_number += 1
        return number

In [ ]:
numbers = OrderNumbers(105)
print(next(numbers), next(numbers))
print(list(islice(numbers, 3)))
print("наступний буде:", numbers.next_number)

`islice(numbers, 3)` узяв рівно три номери — 107, 108, 109 — і зупинився. `list(numbers)` без обмеження не закінчився б ніколи. Курсор пам'ятає, що далі 110.

### Приклад 1.9. `iter(функція, стоп)`: читати, доки не прийде маркер

Повідомлення з каналу застосунку отримуємо функцією `receive()`; маркер `"END"` означає кінець зміни. `iter` з двома аргументами викликає функцію, доки вона не поверне маркер.

In [ ]:
incoming = ["18:55 D-2 105 picked", "19:02 D-2 105 delivered", "END", "19:10 D-1 106 picked"]


def receive():
    return incoming.pop(0)

In [ ]:
for message in iter(receive, "END"):
    print(message)
print("лишилось у каналі:", incoming)

Цикл отримав два повідомлення, на `"END"` зупинився — і маркер, і все після нього в цикл не потрапили; повідомлення після маркера лишилось у каналі. Це заміна для `while True: … if message == "END": break`.

### Задача 1. Курсор з `peek()`

| | `Peekable` |
|---|---|
| **Що це?** | курсор над будь-яким ітерабельним, зокрема нескінченним |
| **Знає** | `_it` — внутрішній ітератор; `_buffer` — елемент, який уже підглянули, але не віддали |
| **Вміє** | `next(cursor)`; `for`; `peek(default=None)` — показати наступний елемент, **не забираючи** його |
| **Правила** | `peek` двічі поспіль — той самий елемент; після кінця даних `peek` повертає `default` |
| **Не робить** | не читає все наперед — лише один елемент (інакше не працюватиме з `count()`) |

Допиши `__iter__`, `__next__` і `peek`. Перевірка — у наступній клітинці.

In [ ]:
from itertools import count


class Peekable:
    def __init__(self, iterable):
        self._it = iter(iterable)
        self._buffer = []

    # YOUR CODE HERE
    pass


In [ ]:
cursor = Peekable(LOG[2:5])
print(cursor.peek())
assert cursor.peek() == "18:03 D-1 101 picked" == next(cursor)
assert next(cursor) == "18:05 D-2 102 picked"
assert cursor.peek() == "18:07 D-3 103 picked"
assert list(cursor) == ["18:07 D-3 103 picked"]
assert cursor.peek() is None and cursor.peek("кінець") == "кінець"
numbers = Peekable(count())
assert numbers.peek() == 0 and next(numbers) == 0 and next(numbers) == 1
print("✅ Задача 1 розв'язана")

---
## 2. Генератор зсередини

Генераторна функція при виклику **не виконується**, а створює об'єкт-генератор. Модуль `inspect` показує, в якому він стані:

```text
gen = shift()      GEN_CREATED    тіло ще не почалось
next(gen)    ──►   GEN_RUNNING    виконується до yield
yield        ──►   GEN_SUSPENDED  пауза, локальні змінні збережені
next / send  ──►   GEN_RUNNING    далі від того самого yield
return / кінець / close()  ──►  GEN_CLOSED
```

In [ ]:
from inspect import getgeneratorstate


def shift():
    print("зміна почалась")
    yield "перша подія"
    print("між подіями")
    yield "друга подія"
    print("зміна закінчилась")
    return "звіт готовий"

### Приклад 2.1. Виклик нічого не виконує

In [ ]:
gen = shift()
print(type(gen).__name__, getgeneratorstate(gen), iter(gen) is gen)

«зміна почалась» **не** надрукувалось: тіло ще не почало виконуватись. Генератор — це ітератор (`iter(gen) is gen`), тому він одноразовий.

### Приклад 2.2. Стани й `return`

🔮 **Прогноз:** коли надрукується «між подіями»? Куди подінеться «звіт готовий»?

In [ ]:
gen = shift()
print(getgeneratorstate(gen))
print(next(gen))
print(getgeneratorstate(gen))
print(next(gen))
try:
    next(gen)
except StopIteration as stop:
    print("StopIteration:", stop.value)
print(getgeneratorstate(gen))

<details>
<summary>Відповідь</summary>

З другим `next()`: кожен `next()` виконує тіло **від попереднього `yield` до наступного**. `return` не віддає значення як елемент, а кладе його в `StopIteration.value`. Закритий генератор уже нічого не віддасть.

</details>

### Приклад 2.3. `for` і `list` гублять `return`

In [ ]:
print(list(shift()))

Усі три повідомлення надрукувались, у списку — дві події, а «звіт готовий» зник: `list` і `for` ловлять `StopIteration` мовчки. Забрати `return`-значення вміє `yield from` (приклад 3.8).

### Приклад 2.4. `close()` і `finally`: прибирання за собою

Генератор, що читає з мережі чи файлу, мусить закрити з'єднання, навіть якщо споживач узяв лише частину даних.

In [ ]:
def read_events(lines):
    print("відкрили з'єднання")
    try:
        for line in lines:
            yield line
    finally:
        print("закрили з'єднання")

In [ ]:
events = read_events(LOG)
print(next(events))
print(next(events))
print(next(events))
print(next(events))
print(next(events))
print(next(events))
print(next(events))
events.close()
print(getgeneratorstate(events))

Сім `next` — сім рядків журналу, а перед першим — «відкрили з'єднання». Далі ми самі закрили генератор: `close()` кидає всередину, на місці паузи, виняток `GeneratorExit`; спрацьовує `finally` — «закрили з'єднання», і стан — `GEN_CLOSED`. Решта чотири рядки так і не прочитані.

### Приклад 2.5. `break` у `for` і `contextlib.closing`

🔮 **Прогноз:** що надрукується раніше — «закрили з'єднання» чи «після циклу»?

In [ ]:
from contextlib import closing

for line in read_events(LOG):
    print(line)
    if "picked" in line:
        break
print("після циклу")

with closing(read_events(LOG)) as events:
    print(next(events))
print("після with")

<details>
<summary>Відповідь</summary>

«закрили з'єднання» — раніше. Після `break` на генератор не лишилось посилань, і CPython одразу закрив його, запустивши `finally`. Але це деталь реалізації: в інших версіях Python закриття може статися пізніше. Надійно — `with closing(генератор)`: `close()` гарантовано викликається на виході з `with`.

</details>

---
## 3. `.send()`: генератор, що приймає дані

Досі дані текли **з** генератора. Метод `.send(value)` передає значення **в** генератор: воно стає результатом виразу `yield`. Такий генератор називають **сопрограмою** (coroutine).

### Приклад 3.1. Середній час доставки на льоту

🔮 **Прогноз:** що надрукують `next(avg)` і чотири `send`?

In [ ]:
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        minutes = yield average
        total += minutes
        count += 1
        average = round(total / count, 1)


In [ ]:
avg = running_average()
print(next(avg))
for minutes in [18, 24, 37, 21]:
    print(avg.send(minutes))

<details>
<summary>Відповідь</summary>

`None`, а далі середнє після кожної доставки: 18.0, 21.0, 26.3, 25.0. Рядок `minutes = yield average` робить дві речі: віддає назовні поточне середнє і ставить функцію на паузу, **чекаючи** наступне число. Стан — `total` і `count` — живе в локальних змінних, без класу й глобальних змінних.

</details>

### Приклад 3.2. `send` крок за кроком

Сопрограма-луна друкує, що отримала. Подивимось, у якому порядку відбувається обмін.

In [ ]:
def echo():
    print("  старт: дійшли до першого yield")
    received = yield "готовий"
    while True:
        print(f"  отримав {received!r}")
        received = yield f"обробив {received}"

In [ ]:
gen = echo()
print(next(gen))
print(gen.send("picked"))
print(gen.send("delivered"))
print(next(gen))

- `next(gen)` довів генератор до **першого** `yield` і забрав його значення «готовий».
- `send("picked")` продовжив роботу: `"picked"` став результатом `yield`, тіло дійшло до наступного `yield` і віддало «обробив picked» — це й повернув `send`.
- `next(gen)` — те саме, що `gen.send(None)`: генератор отримав `None`.

### Приклад 3.3. Чому спершу потрібен `next`

Без запуску генератор ще не стоїть на `yield`, і приймати значення нікому.

In [ ]:
fresh = running_average()
try:
    fresh.send(25)
except TypeError as error:
    print(error)
print(getgeneratorstate(fresh), next(fresh))
for minutes in [50, 30, 40]:
    print(fresh.send(minutes))

Перший `send(25)` упав з `TypeError`, але генератор не зламався — він лишився в стані `GEN_CREATED`. Після `next(fresh)` (повернув `None`) усі `send` працюють: 50.0, 40.0, 40.0. Число 25 загубилось — тому про запуск краще не пам'ятати, а сховати його в декоратор.

### Приклад 3.4. Декоратор `@primed`

Той самий прийом, що в уроці 10: обгортка створює генератор, одразу викликає `next` і повертає вже запущений.

In [ ]:
from functools import wraps


def primed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        gen = func(*args, **kwargs)
        next(gen)
        return gen
    return wrapper


@primed
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        minutes = yield average
        total += minutes
        count += 1
        average = round(total / count, 1)

In [ ]:
avg = running_average()
print(getgeneratorstate(avg), running_average.__name__)
print(avg.send(18), avg.send(24))

Генератор повертається вже на паузі (`GEN_SUSPENDED`), тож перший `send` одразу працює. `@wraps` зберіг ім'я функції.

### Приклад 3.5. Генератор як скінченний автомат

В уроці 22 автомат статусів жив у класі `Order`. Той самий автомат можна записати генератором: стан — локальна змінна, подія приходить через `send`, новий стан віддається через `yield`.

In [ ]:
COURIER_TRANSITIONS = {
    ("вільний", "picked"): "везе",
    ("везе", "delivered"): "вільний",
}


@primed
def courier_state():
    state = "вільний"
    while True:
        event = yield state
        new_state = COURIER_TRANSITIONS.get((state, event))
        if new_state is None:
            print(f"  подія {event} у стані «{state}» — пропускаю")
        else:
            state = new_state

In [ ]:
courier = courier_state()
for event in ["picked", "delivered", "delivered", "picked"]:
    print(event, "→", courier.send(event))

Друга поспіль `delivered` — недопустимий перехід: автомат її пропустив і лишився «вільним».

### Приклад 3.6. Той самий автомат — класом

Для порівняння — клас з методом `handle(event)` і історією переходів.

In [ ]:
class CourierState:
    def __init__(self):
        self.state = "вільний"
        self.history = []

    def handle(self, event):
        new_state = COURIER_TRANSITIONS.get((self.state, event))
        if new_state is None:
            print(f"  подія {event} у стані «{self.state}» — пропускаю")
        else:
            self.history.append(f"{self.state} → {new_state}")
            self.state = new_state
        return self.state

In [ ]:
courier = CourierState()
for event in ["picked", "delivered", "delivered", "picked"]:
    print(event, "→", courier.handle(event))
print(courier.state, courier.history)

Результат той самий. Але в класу стан видно ззовні (`courier.state`), є історія, і легко додати інші дії з іменами (`cancel()`, `pause()`).

**Правило:** генератор — коли вхід один (подія) і вихід один (стан), наприклад лічильник у потоці подій. Клас — коли станом цікавиться багато коду, є кілька дій чи історія (як `Order` з уроку 22).

### Задача 2. Сопрограма-лічильник доставок

`delivery_counter()` з `@primed` приймає через `send` пари `(courier, kind)` і віддає **копію** словника «кур'єр → кількість доставок». Події `picked` не рахуються. Копія — щоб зовнішній код не міг зіпсувати лічильник, змінивши отриманий словник.

Перевірка — у наступній клітинці.

In [ ]:
@primed
def delivery_counter():
    # YOUR CODE HERE
    pass


In [ ]:
counter = delivery_counter()
print(counter.send(("D-1", "picked")))
assert counter.send(("D-1", "delivered")) == {"D-1": 1}
assert counter.send(("D-2", "delivered")) == {"D-1": 1, "D-2": 1}
snapshot = counter.send(("D-1", "delivered"))
assert snapshot == {"D-1": 2, "D-2": 1}
snapshot["D-1"] = 100
assert counter.send(("D-3", "picked")) == {"D-1": 2, "D-2": 1}, "віддавай копію, а не сам словник"
print("✅ Задача 2 розв'язана")

### Приклад 3.7. `yield from`: делегування

В уроці 11 `yield from` віддавав елементи іншого ітерабельного — замість циклу `for … yield`. Склеїмо журнали двох планшетів диспетчерської.

In [ ]:
def merged(*logs):
    for lines in logs:
        yield from lines

In [ ]:
tablet_1 = LOG[2:4]
tablet_2 = LOG[9:]
for line in merged(tablet_1, tablet_2):
    print(line)

Чотири рядки: спершу всі з першого планшета, потім з другого. Для простого склеювання є готовий `itertools.chain` (приклад 4.12), але `yield from` уміє більше — дивись далі.

### Приклад 3.8. `yield from` забирає `return`

Генератор-стадія віддає валідні події, а наприкінці повертає кількість битих рядків. `yield from` для зовнішнього коду — звичайне передавання подій, а сам отримує `return`-значення.

In [ ]:
def valid_events(lines):
    broken = 0
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            broken += 1
            continue
        yield parts
    return broken


def with_report(lines):
    broken = yield from valid_events(lines)
    print(f"битих рядків: {broken}")

In [ ]:
events = list(with_report(LOG))
print(len(events), events[2])

`with_report` віддав 10 подій і дізнався, що битий рядок один. Список `events` знадобиться в розділі 4.

### Приклад 3.9. `yield from` передає й `send`

Середнє за кожне замовлення-пакет: внутрішня сопрограма `averager` рахує, доки не отримає `None`, і повертає середнє. Зовнішня `batch_report` запускає `averager` знову й знову й складає результати. Значення з `send` проходять крізь `yield from` прямо у внутрішню.

In [ ]:
def averager():
    total = 0
    count = 0
    average = None
    while True:
        minutes = yield average
        if minutes is None:
            return average
        total += minutes
        count += 1
        average = round(total / count, 1)


def batch_report(results):
    while True:
        final = yield from averager()
        results.append(final)

In [ ]:
results = []
report_gen = batch_report(results)
next(report_gen)
for minutes in [18, 24, None, 37, 21, None]:
    report_gen.send(minutes)
print(results)

Ми надсилали числа **зовнішньому** генератору, а рахував **внутрішній**: `None` завершив перший `averager` з результатом 21.0, `batch_report` записав його й запустив новий — 29.0 для другого пакета. Саме з `yield from` і сопрограм на генераторах виросли `async` / `await` (урок 28).

---
## 4. `itertools`

Модуль `itertools` — набір «цеглинок», з яких збирають конвеєри без ручних циклів. В уроці 11 були `count`, `chain`, `islice`. Далі — ті, що розв'язують задачі диспетчерської.

### Приклад 4.1. `dropwhile` і `takewhile`: межі потоку

Журнал упорядкований за часом. Треба відкинути події до початку зміни й узяти першу половину зміни. Рядки порівнюються як текст, а формат «ГГ:ХХ» сортується так само, як час.

🔮 **Прогноз:** скільки подій у першій половині зміни (18:00–18:30)?

In [ ]:
from itertools import dropwhile, takewhile

shift_events = dropwhile(lambda line: line < "18:00", LOG)
print(next(shift_events))

first_half = takewhile(lambda line: line < "18:30", dropwhile(lambda line: line < "18:00", LOG))
print(len(list(first_half)))

<details>
<summary>Відповідь</summary>

6, включно з битим рядком 18:24 — він теж «рядок до 18:30». Биті рядки відкидає розбір, а не межі часу.

</details>

- `dropwhile` відкидає елементи, **доки** умова правдива, а потім пропускає все, вже не перевіряючи;
- `takewhile` бере елементи, **доки** умова правдива, і на першому хибному **зупиняє** потік.

### Приклад 4.2. `takewhile` проти `filter` на нескінченному потоці

In [ ]:
from itertools import count, dropwhile, takewhile

print(list(takewhile(lambda number: number < 110, count(105))))

even = filter(lambda number: number % 2 == 0, count(105))
print(list(islice(even, 3)))

`takewhile` сам зупинився на 110. `filter` не знає, що потік упорядкований, і перевіряє **всі** елементи — `list(filter(…, count()))` не закінчився б ніколи, тому межу йому поставив `islice`.

### Приклад 4.3. Ціна `takewhile`: подія, що запізнилась

Подія 18:10 прийшла пізніше за 18:31 (застосунок кур'єра був без зв'язку).

In [ ]:
late_log = [
    "18:03 D-1 101 picked",
    "18:21 D-1 101 delivered",
    "18:31 D-1 104 picked",
    "18:10 D-3 103 picked",
]
print(list(takewhile(lambda line: line < "18:30", late_log)))
print([line for line in late_log if line < "18:30"])

`takewhile` зупинився на 18:31 і подію 18:10 не побачив. Фільтр перевірив усі рядки й знайшов її. `dropwhile` / `takewhile` покладаються на **упорядкованість**: для потоку, де порядок не гарантований, потрібен фільтр або сортування.

### Приклад 4.4. `groupby`: групи сусідів

`groupby(дані, key)` збирає в групу **сусідні** елементи з однаковим ключем. `events` — 10 подій з прикладу 3.8.

🔮 **Прогноз:** скільки груп дасть `groupby` за кур'єром на несортованих подіях?

In [ ]:
from itertools import groupby

groups = [(courier, len(list(group))) for courier, group in groupby(events, key=lambda event: event[1])]
print(len(groups), groups[:4])

<details>
<summary>Відповідь</summary>

9: `groupby` групує лише сусідні елементи.

</details>

### Приклад 4.5. Спершу сортувати за тим самим ключем

In [ ]:
by_courier = sorted(events, key=lambda event: event[1])
print([(courier, len(list(group))) for courier, group in groupby(by_courier, key=lambda event: event[1])])

По групі на кур'єра: D-1 — 5 подій, D-2 — 3, D-3 — 2. Але сортування потребує **всіх** даних у пам'яті. `groupby` доречний, коли дані **вже** прийшли згруповані: журнал, відсортований за днем, файл, розбитий за кур'єром.

### Приклад 4.6. Пастка: група читається лише один раз

Групи `groupby` — не списки, а «вікна» в той самий потік. Що буде, якщо спершу зібрати всі групи, а потім читати їх?

In [ ]:
groups = list(groupby(by_courier, key=lambda event: event[1]))
print([(courier, list(group)) for courier, group in groups])

Усі групи порожні: щоб перейти до наступного ключа, `groupby` прочитав (і викинув) елементи попередньої групи. Групу треба використати **одразу** в тому самому циклі — `len(list(group))`, як у прикладі 4.5.

### Приклад 4.7. Для змішаного потоку — `Counter`

Порахувати події кожного кур'єра можна й без сортування — один прохід, пам'ять лише на ключі (урок 6).

In [ ]:
from collections import Counter

print(Counter(event[1] for event in events))

Той самий результат, що й `sorted` + `groupby`, але без сортування.

### Приклад 4.8. `pairwise` і `accumulate`

Найдовша пауза між подіями — сигнал, що застосунок втрачав зв'язок. `pairwise` (Python 3.10+) віддає пари сусідніх елементів, `accumulate` — наростаючий підсумок. Спершу функція, що переводить «ГГ:ХХ» у хвилини від півночі.

In [ ]:
from itertools import accumulate, pairwise


def minutes(hhmm):
    hours, mins = hhmm.split(":")
    return int(hours) * 60 + int(mins)

In [ ]:
times = [minutes(event[0]) for event in events]
gaps = [later - earlier for earlier, later in pairwise(times)]
print(gaps, max(gaps))

delivered = [1 if event[3] == "delivered" else 0 for event in events]
print(list(accumulate(delivered)))

Найдовша пауза — 14 хв (між 18:07 і 18:21). `accumulate` показує, скільки доставок уже виконано після кожної події: наприкінці 6.

### Приклад 4.9. `accumulate` з функцією й стартовим значенням

In [ ]:
fares = [230, 150, 270, 180, 120, 410]
print(list(accumulate(fares)))
print(list(accumulate(fares, max)))
print(list(accumulate(fares, initial=0)))

Без функції — сума наростаючим підсумком. З `max` — найдорожча поїздка «на цей момент»: 230, 230, 270, … 410. `initial=0` додає стартове значення на початок — результат на один елемент довший.

### Приклад 4.10. `combinations`: перебір без вкладених циклів

В уроці 23 ми шукали набори поїздок на ваучер 500 грн перебором з поверненням. Для наборів фіксованого розміру є готовий перебір.

In [ ]:
from itertools import combinations

fares = [230, 150, 270, 180, 120, 410]
print([pair for pair in combinations(fares, 2) if sum(pair) == 500])
print([trio for trio in combinations(fares, 3) if sum(trio) == 500])
print(sum(1 for size in range(1, len(fares) + 1) for _ in combinations(fares, size)))

Пара (230, 270) і трійка (230, 150, 120) — ті самі набори, що в уроці 23. Але `combinations` перебирає **всі** 63 непорожні підмножини — без відсікання. Для 6 поїздок це дрібниця; для 40 — понад трильйон.

### Приклад 4.11. `product` і `permutations`

`product` — «кожен з кожним» (вкладені цикли), `permutations` — усі порядки.

In [ ]:
from itertools import permutations, product

couriers = ["D-1", "D-2"]
districts = ["Поділ", "Оболонь", "Центр"]
assignments = list(product(couriers, districts))
print(len(assignments), assignments[:3])

routes = list(permutations(districts))
print(len(routes), routes[0], routes[-1])

2 кур'єри × 3 райони = 6 призначень, як два вкладені `for`. Порядків об'їзду трьох районів — 3! = 6. Для 10 районів їх уже 3 628 800: повний перебір росте дуже швидко.

### Приклад 4.12. `chain`, `chain.from_iterable`, `zip_longest`

In [ ]:
from itertools import chain, zip_longest

day_shift = ["D-1", "D-2"]
night_shift = ["D-3"]
print(list(chain(day_shift, night_shift)))

orders_by_courier = [[101, 104], [102], [103]]
print(list(chain.from_iterable(orders_by_courier)))

print(list(zip(day_shift, night_shift)))
print(list(zip_longest(day_shift, night_shift, fillvalue="—")))

- `chain(a, b)` — склеїти кілька потоків, як `merged` з прикладу 3.7;
- `chain.from_iterable` — сплющити потік списків;
- `zip` обрізав пару до найкоротшого — D-2 загубився; `zip_longest` доповнив коротший потік значенням `fillvalue`.

### Приклад 4.13. `tee`: два проходи по одному ітератору

Ітератор одноразовий. `tee` розгалужує його на кілька незалежних копій.

In [ ]:
from itertools import tee

stream = iter(times)
for_gaps, for_count = tee(stream, 2)
print(max(later - earlier for earlier, later in pairwise(for_gaps)))
print(sum(1 for _ in for_count))

Одна копія порахувала найдовшу паузу (14), друга — кількість подій (10). Ціна: поки одна копія попереду, `tee` тримає в пам'яті все, що друга ще не прочитала. Якщо копії розходяться далеко, простіше зробити `list`.

### Приклад 4.14. `islice` на списку і на ітераторі

Цей приклад — ключ до Задачі 3.

In [ ]:
print(list(islice(LOG, 2)))
print(list(islice(LOG, 2)))

cursor = iter(LOG)
print(list(islice(cursor, 2)))
print(list(islice(cursor, 2)))

На **списку** `islice` щоразу починає спочатку — двічі ті самі два рядки. На **одному ітераторі** кожен `islice` продовжує з того місця, де зупинився попередній: другий виклик дав 3-й і 4-й рядки.

### Задача 3. Пачки: `batched`

`batched(iterable, n)` віддає кортежі по `n` елементів, останній може бути коротшим. Має працювати з **нескінченним** потоком: `next(batched(count(), 2))` → `(0, 1)`. (У Python 3.12+ є `itertools.batched`; тут пишемо свій.)

Підказка: один ітератор `iter(iterable)` і `tuple(islice(…, n))` у циклі (приклад 4.14); порожня пачка — сигнал зупинитися. Перевірка — у наступній клітинці.

In [ ]:
def batched(iterable, n):
    # YOUR CODE HERE
    pass


In [ ]:
print(list(batched(range(7), 3)))
assert list(batched(range(7), 3)) == [(0, 1, 2), (3, 4, 5), (6,)]
assert list(batched([], 3)) == []
assert next(batched(count(), 2)) == (0, 1)
assert [len(batch) for batch in batched(LOG, 4)] == [4, 4, 3]
print("✅ Задача 3 розв'язана")

---
## 5. ETL-конвеєр: середній час доставки по кур'єрах

**ETL** (extract — transform — load): витягти дані, перетворити, завантажити результат. Кожна стадія — функція «ітерабельне → ітерабельне», і стадії з'єднуються як труби:

```text
LOG ──► parse ──► in_shift ──► durations ──► report
 рядки   рядок →    dropwhile    пара           середнє
         подія      до 18:00     picked →       по кур'єрах
           │                     delivered
           └──► rejected (биті рядки — окремо, не губимо мовчки)
```

### Приклад 5.1. Стадії конвеєра

In [ ]:
def parse(lines, rejected):
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            rejected.append(line)
            continue
        time, courier, order, kind = parts
        yield {"time": minutes(time), "courier": courier, "order": int(order), "kind": kind}


def in_shift(events, start):
    return dropwhile(lambda event: event["time"] < start, events)


def durations(events):
    picked = {}
    for event in events:
        if event["kind"] == "picked":
            picked[event["order"]] = event["time"]
        elif event["kind"] == "delivered" and event["order"] in picked:
            yield event["courier"], event["time"] - picked.pop(event["order"])


def report(pairs):
    by_courier = {}
    for courier, spent in pairs:
        by_courier.setdefault(courier, []).append(spent)
    return {courier: round(sum(spent) / len(spent), 1) for courier, spent in sorted(by_courier.items())}

In [ ]:
rejected = []
pipeline = durations(in_shift(parse(LOG, rejected), minutes("18:00")))
print(report(pipeline))
print(rejected)

- `parse` перетворює рядок на словник-подію, биті рядки відкладає в `rejected`.
- `in_shift` — одна `dropwhile`: доставки 97 і 98 попередньої зміни далі не йдуть.
- `durations` пам'ятає лише **відкриті** замовлення: доставлене виходить зі словника через `pop`, тож пам'ять не росте разом з потоком.
- `report` — єдине місце, де дані накопичуються.

`D-1` доставив два замовлення: 101 за 18 хв і 104 за 21 хв — середнє 19.5.

### Приклад 5.2. Конвеєр лінивий

Рядок, що з'єднує стадії, нічого не обчислює. Подивимось, коли насправді читається журнал — за списком `rejected`.

In [ ]:
rejected = []
pipeline = durations(in_shift(parse(LOG, rejected), minutes("18:00")))
print(type(pipeline).__name__, rejected)
print(next(pipeline), rejected)
print(list(pipeline), rejected)

Одразу після з'єднання `rejected` порожній: журнал ще ніхто не читав. Перший `next` протягнув через труби рівно стільки рядків, скільки треба для першої тривалості, — до 18:21; битий рядок 18:24 ще попереду. Лише коли `list` дочитав потік, битий рядок потрапив у `rejected`.

### Приклад 5.3. Кожну стадію перевіряємо окремо

Стадія приймає будь-яке ітерабельне — для перевірки досить списку з двох-трьох рядків. Так ми писатимемо тести в уроці 26.

In [ ]:
sample = ["18:00 D-9 1 picked", "18:10 D-9 1 delivered", "битий"]
bad = []
parsed = list(parse(sample, bad))
assert parsed[0] == {"time": 1080, "courier": "D-9", "order": 1, "kind": "picked"}
assert bad == ["битий"]
assert list(in_shift(parsed, minutes("18:05"))) == parsed[1:]
assert list(durations(parsed)) == [("D-9", 10)]
assert list(durations(parsed[1:])) == [], "доставка без picked не рахується"
assert report([("D-9", 10), ("D-9", 20)]) == {"D-9": 15.0}
print("усі стадії перевірено окремо")

Жодна перевірка не потребувала всього журналу: кожна стадія отримала крихітний вхід, і ми точно знаємо, що вона має повернути.

### Приклад 5.4. Нова стадія: запізнення

Пропустити далі лише доставки, довші за `limit` хвилин. Стадію ставимо між `durations` і `list`, не змінюючи інших функцій.

In [ ]:
def late(pairs, limit):
    for courier, spent in pairs:
        if spent > limit:
            yield courier, spent

In [ ]:
print(list(late(durations(in_shift(parse(LOG, []), minutes("18:00"))), 30)))
print(list(late(durations(in_shift(parse(LOG, []), minutes("18:00"))), 20)))

Довше 30 хв — лише замовлення D-3 (37 хв); довше 20 хв — ще D-2 (24) і D-1 (21). Нова вимога — нова маленька стадія, решта конвеєра не змінилась.

### Приклад 5.5. Знайди помилку: дві цифри з одного генератора

🔮 **Прогноз:** що надрукує перший `print`?

In [ ]:
parsed = parse(LOG, [])
total = sum(1 for _ in parsed)
print(total, report(durations(parsed)))

parsed = list(parse(LOG, []))
print(len(parsed), report(durations(parsed)))

<details>
<summary>Відповідь</summary>

`10 {}`: `sum` вичерпав генератор `parsed`, і `report` отримав порожній потік. Виправлення — створити конвеєр заново або, якщо даних мало, один раз зробити `list(parse(…))`: список можна обходити скільки завгодно разів.

</details>

### Приклад 5.6. Ковзне середнє

`moving_average(values, k)` — середнє останніх `k` значень для кожної позиції, починаючи з `k`-ї. Суму вікна не перераховуємо щоразу (ковзне вікно з уроку 12): додаємо нове значення й віднімаємо те, що випало. Працює з будь-яким ітерабельним, зокрема генератором.

In [ ]:
from collections import deque


def moving_average(values, k):
    window = deque()
    window_sum = 0
    for value in values:
        window.append(value)
        window_sum += value
        if len(window) > k:
            window_sum -= window.popleft()
        if len(window) == k:
            yield window_sum / k

In [ ]:
print(list(moving_average([18, 24, 37, 21], 2)))
spent = (value for _, value in durations(in_shift(parse(LOG, []), minutes("18:00"))))
print(list(moving_average(spent, 2)))
assert list(moving_average([10, 20, 30], 3)) == [20.0]
assert list(moving_average([10, 20], 3)) == []

Для `[18, 24, 37, 21]` і `k=2`: (18+24)/2 = 21.0, (24+37)/2 = 30.5, (37+21)/2 = 29.0. Тривалості з конвеєра — ті самі 18, 24, 37, 21, тож і результат той самий, хоч на вході генератор. Якщо значень менше за `k`, середніх немає.

### Приклад 5.7. Підсумок зміни через `groupby`

«Кур'єр → (скільки забрав, скільки доставив)» для списку подій `events` (приклад 3.8): `sorted` + `groupby`, як у прикладі 4.5.

In [ ]:
def shift_summary(events):
    summary = {}
    ordered = sorted(events, key=lambda event: event[1])
    for courier, group in groupby(ordered, key=lambda event: event[1]):
        kinds = [event[3] for event in group]
        summary[courier] = (kinds.count("picked"), kinds.count("delivered"))
    return summary

In [ ]:
print(shift_summary(events))
assert shift_summary(events) == {"D-1": (2, 3), "D-2": (1, 2), "D-3": (1, 1)}
assert shift_summary([]) == {}

У D-1 доставок (3) більше, ніж «забрав» (2): одна доставка — 97 з попередньої зміни. Групу ми одразу перетворили на список `kinds` — це обхід пастки з прикладу 4.6.

### Приклад 5.8. Pull і push: середнє після кожної доставки

У конвеєрі з генераторів дані **тягне** споживач (pull): `report` просить наступну тривалість, `durations` — наступну подію, і так до джерела. Сопрограми працюють навпаки — дані **штовхає** джерело (push):

```text
pull (генератори):  report ──next()──► durations ──next()──► in_shift ──► parse ──► LOG
push (сопрограми):  подія прийшла ──send()──► live_durations ──send()──► running_average
```

Диспетчерська хоче бачити середнє **після кожної** доставки, щойно подія прийшла, — це задача push.

In [ ]:
@primed
def live_durations(on_done):
    picked = {}
    while True:
        event = yield
        if event["kind"] == "picked":
            picked[event["order"]] = event["time"]
        elif event["order"] in picked:
            on_done(event["courier"], event["time"] - picked.pop(event["order"]))


def show_average(courier, spent):
    print(f"  {courier}: {spent} хв, середнє зміни {live_average.send(spent)}")

In [ ]:
live_average = running_average()
sink = live_durations(show_average)
for event in in_shift(parse(LOG, []), minutes("18:00")):
    sink.send(event)

Кожна подія сама «проштовхнулась» у `live_durations`; щойно замовлення доставлене, тривалість пішла в `running_average` з прикладу 3.4, і середнє оновилось: 18.0, 21.0, 26.3, 25.0 — ті самі числа, що в прикладі 3.1.

| Підхід | Пам'ять | Проходів | Хто керує | Коли |
|---|---|---|---|---|
| списки між стадіями | весь проміжний результат | скільки завгодно | код, що викликає стадії | мало даних, треба кілька проходів чи індекси |
| конвеєр генераторів | один елемент на стадію | один | споживач (pull) | великі файли, потоки, звіти |
| клас-ітератор | як генератор | один на курсор | споживач | курсор з власними методами: `peek`, позиція |
| сопрограма `.send()` | стан сопрограми | — | джерело (push) | реакція на кожну подію, автомат, ковзні підсумки |

## ✅ Самоперевірка

1. Чим ітерабельне відрізняється від ітератора? Що повертає `__iter__` кожного з них?
2. Куди потрапляє `return` генератора і хто його забирає?
3. Навіщо запускати генератор перед першим `send`?
4. Чому `takewhile` працює з нескінченним потоком, а `filter` — ні?
5. Коли `groupby` дає неправильні групи?
6. Чому кожну стадію ETL-конвеєра легко тестувати?
7. Чому групи `groupby` порожні, якщо спершу зробити `list(groupby(…))`?
8. Чим pull-конвеєр генераторів відрізняється від push-конвеєра сопрограм?

<details>
<summary>Відповіді</summary>

1. Ітерабельне на кожен `iter()` дає новий курсор; ітератор — сам курсор, його `__iter__` повертає `self`.
2. У `StopIteration.value`; його отримує `yield from`.
3. Щоб генератор дійшов до першого `yield`, який прийме значення.
4. `takewhile` зупиняється на першому хибному елементі; `filter` перевіряє всі.
5. Коли однакові ключі не йдуть підряд — дані треба спершу відсортувати.
6. Стадія приймає будь-яке ітерабельне: для тесту досить списку з кількох рядків.
7. Переходячи до наступного ключа, `groupby` читає й викидає елементи попередньої групи — групу треба використати одразу.
8. У pull споживач просить наступний елемент (`next`); у push джерело саме передає кожну подію (`send`), щойно вона прийшла.

</details>

### Шпаргалка

```python
class Cursor:                          # клас-ітератор
    def __iter__(self): return self
    def __next__(self): ...            # елемент або raise StopIteration

class Log:                             # ітерабельне: новий курсор щоразу
    def __iter__(self):
        yield from self.lines

value = yield result                   # сопрограма: send(value)
gen = coro(); next(gen); gen.send(x)   # або декоратор @primed
result = yield from sub()              # делегування + return-значення

dropwhile(pred, it)  takewhile(pred, it)  groupby(sorted(it, key=k), key=k)
pairwise(it)  accumulate(it)  combinations(it, r)  islice(it, n)
product(a, b)  permutations(it)  chain(a, b)  zip_longest(a, b)  tee(it, 2)
iter(func, sentinel)                   # викликати func, доки не поверне sentinel
```

## Далі

- **Урок 26 — Тестування з pytest**: стадії конвеєра — перші кандидати на тести.
- **Урок 28 — asyncio**: `async` / `await` виросли з `.send()` і `yield from`.